# Quantum-Enhanced ML vs Classical ML — Crop Yield Prediction

This notebook is designed to make a **fair comparison** between a classical ML model
and a **hybrid quantum-enhanced ML model**.

### Models

1. Linear Regression
2. Random Forest
3. Quantum Kernel Residual Model:
   - Train the classical model first.
   - Calculate its residuals.
   - Use a quantum fidelity kernel to learn the residual pattern.
   - Tune quantum regularization (`lambda`) on a validation set.
   - Tune the quantum correction strength (`blend`) on the validation set.
   - Evaluate the final model once on the untouched test set.

### Important

The notebook does **not** force the quantum model to win.

If the quantum model does not improve the classical model on the test set,
the notebook reports that honestly.

The validation set is used for choosing hyperparameters, so the test set remains
a fair final comparison.

In [1]:
# ============================================================
# 1. IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import time
import warnings

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

from qiskit.circuit.library import ZZFeatureMap
from qiskit.primitives import StatevectorSampler
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_algorithms.state_fidelities import ComputeUncompute

np.random.seed(42)

print("Imports successful.")
print("Qiskit version:", __import__("qiskit").__version__)

Imports successful.
Qiskit version: 2.5.0


In [2]:
# ============================================================
# 2. LOAD DATA
# ============================================================

DATA_PATH = "Crop Yiled with Soil and Weather.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

feature_cols = ["Fertilizer", "temp", "N", "P", "K"]
target_col = "yeild"

X = df[feature_cols].values.astype(float)
y = df[target_col].values.astype(float)

print("\nFeatures:", feature_cols)
print("Target:", target_col)
print("Missing values:")
print(df[feature_cols + [target_col]].isnull().sum())

Dataset shape: (2596, 6)


,Fertilizer,temp,N,P,K,yeild
0,80.0,28.0,80.0,24.0,20.0,12.0
1,77.0,27.0,78.0,23.0,20.0,12.0
2,80.0,26.0,80.0,24.0,20.0,12.0
3,80.0,28.0,80.0,24.0,20.0,12.0
4,78.0,27.0,78.0,23.0,19.0,12.0



Features: ['Fertilizer', 'temp', 'N', 'P', 'K']
Target: yeild
Missing values:
Fertilizer    0
temp          0
N             0
P             0
K             0
yeild         0
dtype: int64


In [3]:
# ============================================================
# 3. TRAIN / VALIDATION / TEST SPLIT
# ============================================================
# 64% train
# 16% validation
# 20% final test
#
# The final test set is NOT used for hyperparameter selection.

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.20,
    random_state=42
)

scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (1660, 5)
Validation: (416, 5)
Test: (520, 5)


In [4]:
# ============================================================
# 4. METRIC FUNCTION
# ============================================================

def metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)

    return {
        "R2": r2_score(y_true, y_pred),
        "RMSE": np.sqrt(mse),
        "MAE": mean_absolute_error(y_true, y_pred)
    }


def print_metrics(name, y_true, y_pred):
    m = metrics(y_true, y_pred)

    print(f"\n{name}")
    print("-" * 40)
    print(f"R2   : {m['R2']:.6f}")
    print(f"RMSE : {m['RMSE']:.6f}")
    print(f"MAE  : {m['MAE']:.6f}")

    return m

## 5. Classical Baselines

We first train the classical models normally.

These predictions are the baseline that the quantum model is trying to improve.

In [5]:
# ============================================================
# 5A. LINEAR REGRESSION
# ============================================================

lr = LinearRegression()

start = time.perf_counter()
lr.fit(X_train_s, y_train)
lr_train_time = time.perf_counter() - start

lr_val_pred = lr.predict(X_val_s)
lr_test_pred = lr.predict(X_test_s)

lr_val_metrics = print_metrics(
    "Linear Regression - Validation",
    y_val,
    lr_val_pred
)

lr_test_metrics = print_metrics(
    "Linear Regression - Test",
    y_test,
    lr_test_pred
)


Linear Regression - Validation
----------------------------------------
R2   : 0.873007
RMSE : 0.688323
MAE  : 0.567240

Linear Regression - Test
----------------------------------------
R2   : 0.862834
RMSE : 0.717989
MAE  : 0.583931


In [6]:
# ============================================================
# 5B. RANDOM FOREST
# ============================================================

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

start = time.perf_counter()
rf.fit(X_train_s, y_train)
rf_train_time = time.perf_counter() - start

rf_val_pred = rf.predict(X_val_s)
rf_test_pred = rf.predict(X_test_s)

rf_val_metrics = print_metrics(
    "Random Forest - Validation",
    y_val,
    rf_val_pred
)

rf_test_metrics = print_metrics(
    "Random Forest - Test",
    y_test,
    rf_test_pred
)


Random Forest - Validation
----------------------------------------
R2   : 0.991577
RMSE : 0.177265
MAE  : 0.114003

Random Forest - Test
----------------------------------------
R2   : 0.990543
RMSE : 0.188524
MAE  : 0.125211


## 6. Choose the Classical Baseline

For the main hybrid experiment, we use the better validation model.

The quantum model will learn the **residuals** of this model rather than replacing
the classical model completely.

This is important because the quantum circuit does not have to relearn the entire
crop-yield relationship.

In [7]:
# ============================================================
# 6. SELECT CLASSICAL BASELINE
# ============================================================

if rf_val_metrics["RMSE"] <= lr_val_metrics["RMSE"]:
    baseline_name = "Random Forest"
    baseline_model = rf
    baseline_val_pred = rf_val_pred
    baseline_test_pred = rf_test_pred
else:
    baseline_name = "Linear Regression"
    baseline_model = lr
    baseline_val_pred = lr_val_pred
    baseline_test_pred = lr_test_pred

print("Selected baseline:", baseline_name)
print("Validation RMSE:", metrics(y_val, baseline_val_pred)["RMSE"])

Selected baseline: Random Forest
Validation RMSE: 0.17726538395251418


## 7. Quantum Kernel

We use a 5-qubit `ZZFeatureMap`.

The quantum kernel measures similarity between samples after encoding their
classical features into quantum states.

For computational reasons, only a subset of the training data is used as
quantum reference points.

This is the expensive section of the notebook.

In [8]:
# ============================================================
# 7. QUANTUM KERNEL SETUP
# ============================================================

N_Q_TRAIN = 300

# Keep the quantum reference set fixed and reproducible.
rng = np.random.default_rng(42)

q_indices = rng.choice(
    len(X_train_s),
    size=min(N_Q_TRAIN, len(X_train_s)),
    replace=False
)

X_q_train = X_train_s[q_indices]
y_q_train = y_train[q_indices]

# Residuals of the classical baseline on quantum reference samples
baseline_q_train_pred = baseline_model.predict(X_q_train)
residual_q_train = y_q_train - baseline_q_train_pred

print("Quantum reference samples:", X_q_train.shape)
print("Residual std:", np.std(residual_q_train))

n_qubits = X_train_s.shape[1]

feature_map = ZZFeatureMap(
    feature_dimension=n_qubits,
    reps=2,
    entanglement="full"
)

sampler = StatevectorSampler()

fidelity = ComputeUncompute(
    sampler=sampler
)

quantum_kernel = FidelityQuantumKernel(
    feature_map=feature_map,
    fidelity=fidelity
)

print("Quantum kernel ready.")
print("Qubits:", n_qubits)

Quantum reference samples: (300, 5)
Residual std: 0.0721677974111664
Quantum kernel ready.
Qubits: 5


In [ ]:
# ============================================================
# 8. COMPUTE QUANTUM KERNEL MATRICES
# ============================================================
# K_train:
#     quantum reference samples vs quantum reference samples
#
# K_val:
#     validation samples vs quantum reference samples
#
# K_test:
#     test samples vs quantum reference samples
#
# This can take several minutes with statevector simulation.

print("Computing quantum kernel matrices...")
start = time.perf_counter()

K_train = quantum_kernel.evaluate(
    x_vec=X_q_train
)

K_val = quantum_kernel.evaluate(
    x_vec=X_val_s,
    y_vec=X_q_train
)

K_test = quantum_kernel.evaluate(
    x_vec=X_test_s,
    y_vec=X_q_train
)

kernel_time = time.perf_counter() - start

print(f"Quantum kernel time: {kernel_time:.2f} seconds")
print("K_train:", K_train.shape)
print("K_val:", K_val.shape)
print("K_test:", K_test.shape)

eigvals = np.linalg.eigvalsh(K_train)

print("Kernel min eigenvalue:", eigvals.min())

Computing quantum kernel matrices...


## 9. Tune QKRR Properly

This is the key change from the earlier notebook.

The earlier experiment used:

```text
lambda = 0.001
```

without selecting it.

Here we test several regularization strengths on the **validation set**.

A larger lambda prevents the quantum correction coefficients from exploding.

In [ ]:
# ============================================================
# 9. QKRR LAMBDA SEARCH
# ============================================================

lambda_values = [
    1e-3,
    3e-3,
    1e-2,
    3e-2,
    1e-1,
    3e-1,
    1.0,
    3.0,
    10.0,
    30.0,
    100.0
]

lambda_results = []

for lam in lambda_values:

    # Solve:
    # (K + lambda I) alpha = residual

    alpha = np.linalg.solve(
        K_train + lam * np.eye(len(K_train)),
        residual_q_train
    )

    q_correction_val = K_val @ alpha

    raw_hybrid_val = baseline_val_pred + q_correction_val

    m = metrics(y_val, raw_hybrid_val)

    lambda_results.append({
        "lambda": lam,
        "RMSE": m["RMSE"],
        "MAE": m["MAE"],
        "R2": m["R2"],
        "Correction_STD": np.std(q_correction_val),
        "Alpha_STD": np.std(alpha)
    })

lambda_df = pd.DataFrame(lambda_results)

display(
    lambda_df.sort_values("RMSE")
)

best_lambda = lambda_df.sort_values(
    "RMSE"
).iloc[0]["lambda"]

print("Best lambda:", best_lambda)

## 10. Tune the Quantum Correction Strength

Even a well-regularized QKRR correction does not necessarily need to be added
with strength 1.

We therefore learn a blend:

$$
\hat y_{hybrid}
=
\hat y_{classical}
+
w f_{QKRR}(x)
$$

where `w` is selected using validation data.

Importantly, `w = 0` is allowed.

Therefore, if the quantum correction is not useful, validation will select
the classical model rather than forcing a quantum correction.

In [ ]:
# ============================================================
# 10. BEST QKRR + BLEND SEARCH
# ============================================================

alpha_best = np.linalg.solve(
    K_train + best_lambda * np.eye(len(K_train)),
    residual_q_train
)

q_correction_val = K_val @ alpha_best

blend_values = np.array([
    0.0,
    0.05,
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
    0.60,
    0.75,
    1.00
])

blend_results = []

for w in blend_values:

    pred = baseline_val_pred + w * q_correction_val

    m = metrics(y_val, pred)

    blend_results.append({
        "blend": w,
        "RMSE": m["RMSE"],
        "MAE": m["MAE"],
        "R2": m["R2"],
        "Correction_STD": np.std(w * q_correction_val)
    })

blend_df = pd.DataFrame(blend_results)

display(
    blend_df.sort_values("RMSE")
)

best_blend = blend_df.sort_values(
    "RMSE"
).iloc[0]["blend"]

print("Best quantum blend:", best_blend)

## 11. Final Hybrid Quantum Model

Now the quantum hyperparameters are fixed.

We evaluate on the untouched test set.

No test-set tuning is performed here.

In [ ]:
# ============================================================
# 11. FINAL QUANTUM HYBRID TEST PREDICTION
# ============================================================

q_correction_test = K_test @ alpha_best

hybrid_test_pred = (
    baseline_test_pred
    +
    best_blend * q_correction_test
)

hybrid_test_metrics = print_metrics(
    f"{baseline_name} + Quantum QKRR - Test",
    y_test,
    hybrid_test_pred
)

print("\nQuantum correction statistics:")
print("Raw correction std:", np.std(q_correction_test))
print("Applied correction std:", np.std(best_blend * q_correction_test))
print("Correction min:", np.min(best_blend * q_correction_test))
print("Correction max:", np.max(best_blend * q_correction_test))

## 12. Final Comparison

For RMSE and MAE, **lower is better**.

For R², **higher is better**.

The notebook reports whether the quantum-enhanced model actually improved the
classical baseline.

In [ ]:
# ============================================================
# 12. FINAL COMPARISON
# ============================================================

classical_test_metrics = metrics(
    y_test,
    baseline_test_pred
)

final_comparison = pd.DataFrame({
    "Metric": ["R2", "RMSE", "MAE"],
    "Classical ML": [
        classical_test_metrics["R2"],
        classical_test_metrics["RMSE"],
        classical_test_metrics["MAE"]
    ],
    "Quantum-Enhanced ML": [
        hybrid_test_metrics["R2"],
        hybrid_test_metrics["RMSE"],
        hybrid_test_metrics["MAE"]
    ]
})

display(final_comparison)

rmse_improvement = (
    (classical_test_metrics["RMSE"] - hybrid_test_metrics["RMSE"])
    / classical_test_metrics["RMSE"]
) * 100

mae_improvement = (
    (classical_test_metrics["MAE"] - hybrid_test_metrics["MAE"])
    / classical_test_metrics["MAE"]
) * 100

r2_change = (
    hybrid_test_metrics["R2"]
    - classical_test_metrics["R2"]
)

print(f"\nBaseline model: {baseline_name}")
print(f"Best lambda: {best_lambda}")
print(f"Best quantum blend: {best_blend}")

print(f"\nRMSE improvement: {rmse_improvement:+.2f}%")
print(f"MAE improvement : {mae_improvement:+.2f}%")
print(f"R2 change       : {r2_change:+.6f}")

if (
    hybrid_test_metrics["RMSE"] < classical_test_metrics["RMSE"]
    and hybrid_test_metrics["MAE"] < classical_test_metrics["MAE"]
    and hybrid_test_metrics["R2"] > classical_test_metrics["R2"]
):
    print("\nRESULT: Quantum-enhanced ML improved the classical model.")
else:
    print("\nRESULT: Quantum-enhanced ML did NOT improve the classical model on this test set.")
    print("This is a valid result; do not claim quantum superiority.")

In [ ]:
# ============================================================
# 13. PREDICTION-BY-PREDICTION COMPARISON
# ============================================================

prediction_comparison = pd.DataFrame({
    "Actual": y_test,
    "Classical_Prediction": baseline_test_pred,
    "Quantum_Hybrid_Prediction": hybrid_test_pred,
    "Classical_Absolute_Error":
        np.abs(y_test - baseline_test_pred),
    "Quantum_Absolute_Error":
        np.abs(y_test - hybrid_test_pred)
})

prediction_comparison["Quantum_Better"] = (
    prediction_comparison["Quantum_Absolute_Error"]
    <
    prediction_comparison["Classical_Absolute_Error"]
)

display(
    prediction_comparison.head(20)
)

print(
    "\nQuantum prediction better on "
    f"{prediction_comparison['Quantum_Better'].mean() * 100:.2f}% "
    "of test samples."
)

## 14. Why This Version Is Better Than the Earlier Experiment

The earlier QKRR experiment used a fixed:

```text
lambda = 0.001
```

and directly added the full quantum correction.

That produced very large corrections and made the model much worse.

This notebook instead:

1. Uses a validation set.
2. Searches multiple QKRR regularization values.
3. Measures correction magnitude.
4. Searches the quantum correction strength.
5. Allows the best blend to be zero.
6. Evaluates the final choice only once on the test set.

Therefore the experiment asks a much more useful research question:

> **Can a quantum kernel provide useful additional information beyond the classical model?**

rather than assuming that the quantum correction must improve the result.

In [ ]:
# ============================================================
# 15. SAVE RESULTS
# ============================================================

results = {
    "baseline_model": baseline_name,
    "best_lambda": float(best_lambda),
    "best_quantum_blend": float(best_blend),
    "classical_R2": float(classical_test_metrics["R2"]),
    "classical_RMSE": float(classical_test_metrics["RMSE"]),
    "classical_MAE": float(classical_test_metrics["MAE"]),
    "quantum_R2": float(hybrid_test_metrics["R2"]),
    "quantum_RMSE": float(hybrid_test_metrics["RMSE"]),
    "quantum_MAE": float(hybrid_test_metrics["MAE"]),
    "RMSE_improvement_percent": float(rmse_improvement),
    "MAE_improvement_percent": float(mae_improvement),
    "R2_change": float(r2_change)
}

results_df = pd.DataFrame([results])

display(results_df)

results_df.to_csv(
    "quantum_vs_classical_results.csv",
    index=False
)

print("Results saved to quantum_vs_classical_results.csv")